# Interactive plots with uPlot

[uPlot](https://github.com/leeoniya/uPlot) makes fast interactive time series, also with many points. Needs `pip install "scdata[plot]"` and a notebook (Jupyter).

Controls: drag to zoom, double click to reset, hold shift to grow or contract a selection.

In [ ]:
import logging

import scdata as sc
import smartcitizen_connector
from scdata.tools.custom_logger import set_logger_level

# Only errors: 'WARNING', 'INFO' or 'DEBUG' show what scdata and smartcitizen-connector do
set_logger_level('ERROR')
logging.getLogger('smartcitizen_connector').setLevel('ERROR')

sc.__version__

In [ ]:
import pandas as pd

start = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('3D')).strftime('%Y-%m-%d')
end = (pd.Timestamp.now(tz='UTC') - pd.Timedelta('1D')).strftime('%Y-%m-%d')

# Stored in the scdata cache and replaced on each run (tests: see 01_getting_started)
test = sc.Test(name='EXAMPLES_UPLOT',
               devices=[sc.Device(params=sc.APIParams(id=device_id), options=sc.DeviceOptions(min_date=start))
                        for device_id in [18154, 17592]]).save(overwrite=True)
await test.load()

In [ ]:
# A boolean to plot: temperature above the mean of the series
for device in test.devices:
    device.data['WARM'] = device.data['TEMP'] > device.data['TEMP'].mean()

## One axis

In [ ]:
traces = {'1': {'devices': 'all', 'channel': ['TEMP'], 'subplot': 1, 'side': 'left'}}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'title': 'Temperature', 'ylabel': {1: 'Temperature (degC)'}},
              options={'frequency': '30Min', 'min_date': start, 'max_date': end})

## Two axes

Traces on the `right` side use `ylabel_right`:

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': ['TEMP'], 'subplot': 1, 'side': 'left'},
    '2': {'devices': 'all', 'channel': ['WARM'], 'subplot': 1, 'side': 'right'},
}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'title': 'Two axes',
                          'ylabel': {1: 'Temperature (degC)'}, 'ylabel_right': {1: 'Warm'}},
              options={'frequency': '1Min', 'min_date': start, 'max_date': end})

## A device

Plots of a device do not need `devices` in the traces. When resampling, choose the `resample` method: `mean` (default), `min` or `max`:

In [ ]:
traces = {
    '1': {'channel': ['TEMP'], 'subplot': 1, 'side': 'left'},
    '2': {'channel': ['WARM'], 'subplot': 1, 'side': 'right'},
}

test.get_device(18154).ts_uplot(traces=traces,
                                   formatting={'width': 800, 'title': 'Maximum of each 10 minutes',
                                               'ylabel': {1: 'Temperature (degC)'}, 'ylabel_right': {1: 'Warm'}},
                                   options={'frequency': '10Min', 'resample': 'max', 'min_date': start, 'max_date': end})

## Bands and statistics of several devices

`bands` shows the mean and three standard deviations of the devices of a trace; `mean`, `max` and `min` add those series:

In [ ]:
traces = {'1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1, 'side': 'left', 'extras': ['bands']}}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'title': 'Three sigma bands', 'ylabel': {1: 'Temperature (degC)'}},
              options={'frequency': '1Min', 'min_date': start, 'max_date': end})

In [ ]:
traces = {'1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1, 'side': 'left', 'extras': ['mean', 'max', 'min']}}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'title': 'Mean, maximum and minimum', 'ylabel': {1: 'Temperature (degC)'}},
              options={'frequency': '1Min', 'min_date': start, 'max_date': end})

## Filter by a condition

Plot only the values where another column meets a condition (`==`, `<=`, `>=`, `<`, `>`):

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1, 'side': 'left',
          'filter': {'value': 1, 'relationship': '==', 'col': 'WARM'}},
    '2': {'devices': 'all', 'channel': 'WARM', 'subplot': 1, 'side': 'right'},
}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'title': 'Only warm values',
                          'ylabel': {1: 'Temperature (degC)'}, 'ylabel_right': {1: 'Warm'}},
              options={'frequency': '1Min', 'min_date': start, 'max_date': end})

## Several subplots

In [ ]:
traces = {
    '1': {'devices': 'all', 'channel': 'TEMP', 'subplot': 1},
    '2': {'devices': 'all', 'channel': 'HUM', 'subplot': 2},
    '3': {'devices': 'all', 'channel': 'NOISE_A', 'subplot': 3},
}

test.ts_uplot(traces=traces,
              formatting={'width': 800, 'height': 250, 'padding-bottom': 750, 'title': 'Temperature, humidity and noise',
                          'ylabel': {1: 'Temperature (degC)', 2: 'Humidity (%rh)', 3: 'Noise (dBA)'}},
              options={'frequency': '10Min'})